In [ ]:
import os
import re
import json
import gc
import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

# Paths — edit if needed
BASE_DIR = Path(".")
MODELS_BASE = Path("models")
TEST_CSV = Path("../data/processed/test.csv")
OUTPUT_DIR = Path("results/city_swap_all")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# (name, subfolder, uses_scrubbing)
MODELS = [
    ("1_baseline",               "bert_9classes_final",          False),
    ("2_groupdro",               "bert_gdro_eta01_2ep",          False),
    ("3_scrubbing",              "bert_scrubbing",               True),
    ("4_oversampling",           "bert_oversample_only",         False),
    ("5_focal_loss",             "bert_focal_loss",              False),
    ("6_adversarial",            "bert_adversarial",             False),
    ("7_label_smoothing",        "bert_label_smoothing",         False),
    ("8_attribution_reg",        "bert_attr_reg",                False),
    ("9_combined_scrub_gdro",    "bert_debiased_combo",          True),
    ("10_combined_best",         "combined_scrubbing_groupdro",  True),
]

SWAP_CITIES = ["Москва", "Екатеринбург", "Новосибирск", "Краснодар", "Воронеж"]
BATCH_SIZE = 8
MAX_LENGTH = 128

In [ ]:
CITY_PATTERNS = [
    "санкт-петербург", "нижний новгород", "ростов-на-дону",
    "набережные челны", "магнитогорск", "новосибирск",
    "екатеринбург", "красноярск", "волгоград", "калининград",
    "владивосток", "хабаровск", "ставрополь", "саратов",
    "челябинск", "самара", "казань", "москва", "омск",
    "воронеж", "пермь", "тюмень", "томск", "уфа",
    "тольятти", "барнаул", "иркутск", "пенза", "липецк",
    "кемерово", "сочи", "тверь", "минск", "алматы",
    "симферополь", "ярославль", "ульяновск", "ижевск",
    "оренбург", "мск", "спб", "питер",
]
escaped = [re.escape(c) for c in CITY_PATTERNS]
CITY_RE = re.compile(r'\b(' + '|'.join(escaped) + r')\b', re.IGNORECASE)

CITY_WORDS_SCRUB = [
    "москва", "московская", "московский", "мск",
    "санкт-петербург", "петербург", "спб", "питер", "ленинград",
    "новосибирск", "екатеринбург", "казань", "нижний новгород",
    "челябинск", "самара", "омск", "ростов-на-дону", "уфа",
    "красноярск", "воронеж", "пермь", "волгоград",
    "краснодар", "саратов", "тюмень", "тольятти", "ижевск",
    "барнаул", "ульяновск", "иркутск", "хабаровск", "ярославль",
    "владивосток", "махачкала", "томск", "оренбург", "кемерово",
    "новокузнецк", "рязань", "астрахань", "пенза", "липецк",
    "калининград", "тула", "курск", "ставрополь", "сочи",
    "магнитогорск", "томская", "набережные челны", "тверь",
    "минск", "алматы", "киев", "симферополь",
    "область", "край", "республика", "регион",
]
AGE_WORDS = [
    "пенсионер", "пенсионерка", "пенсия", "пенсионный",
    "студент", "студентка", "выпускник", "выпускница",
    "молодой", "молодая", "junior", "senior",
]
ALL_SENSITIVE = set(w.lower() for w in CITY_WORDS_SCRUB + AGE_WORDS)


def swap_cities_in_text(text, target_city):
    if pd.isna(text):
        return ""
    def replacer(match):
        orig = match.group(0)
        return target_city.capitalize() if orig[0].isupper() else target_city.lower()
    return CITY_RE.sub(replacer, str(text))


def scrub_text(text, mask_token="[MASK]"):
    if pd.isna(text):
        return ""
    result = str(text)
    for word in sorted(ALL_SENSITIVE, key=len, reverse=True):
        result = re.compile(re.escape(word), re.IGNORECASE).sub(mask_token, result)
    return result


def predict_batch(texts, model, tokenizer, device, batch_size=BATCH_SIZE):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
            preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        del enc, outputs
    return np.array(all_preds)

In [ ]:
df = pd.read_csv(TEST_CSV)
print(f"Test resumes: {len(df)}")

In [ ]:
def find_model_files(model_dir):
    model_dir = Path(model_dir)
    if (model_dir / "config.json").exists():
        return "hf", model_dir
    candidates = [sub for sub in model_dir.iterdir()
                  if sub.is_dir() and (sub / "config.json").exists()]
    if candidates:
        for c in candidates:
            if c.name == "final":
                return "hf", c
        return "hf", sorted(candidates)[-1]
    return None, None


def load_model_safe(model_name, model_dir, device):
    fmt, path = find_model_files(model_dir)
    if fmt is None:
        print(f"  No model files found in {model_dir}")
        return None
    try:
        tokenizer = AutoTokenizer.from_pretrained(str(path))
        model, loading_info = AutoModelForSequenceClassification.from_pretrained(
            str(path),
            output_loading_info=True,
        )
        missing_keys = set(loading_info.get("missing_keys", []))
        missing_classifier = any(k.startswith("classifier.") for k in missing_keys)
        if missing_classifier:
            print(f"  Invalid checkpoint for evaluation: missing classifier head {sorted(missing_keys)}")
            return None
        model = model.to(device).eval()
        le = None
        for le_path in [path / "label_encoder.joblib",
                        path.parent / "label_encoder.joblib",
                        model_dir / "label_encoder.joblib"]:
            if le_path.exists():
                le = joblib.load(le_path)
                break
        if le is None:
            print(f"  No label_encoder.joblib found for {model_name}")
            return None
        return {"model": model, "tokenizer": tokenizer, "le": le}
    except Exception as e:
        print(f"  Failed to load: {e}")
        return None

In [ ]:
df["text_scrubbed"] = df["resume_text"].apply(scrub_text)
print("Scrubbing done")

In [ ]:
mapping = pd.read_csv(Path("../data/processed/label_to_supercategory_v1.csv"))
l2s = dict(zip(mapping["label"], mapping["supercategory"]))

In [ ]:
all_results = {}

for model_name, model_subdir, uses_scrub in MODELS:
    model_dir = MODELS_BASE / model_subdir
    print(f"\n{'='*60}")
    print(f"{model_name} ({model_subdir})")

    if not model_dir.exists():
        print("  Directory not found, skipping")
        all_results[model_name] = {"error": "directory not found"}
        continue

    loaded = load_model_safe(model_name, model_dir, device)
    if loaded is None:
        all_results[model_name] = {"error": "failed to load"}
        continue

    model = loaded["model"]
    tokenizer = loaded["tokenizer"]
    le = loaded["le"]
    class_names = le.classes_

    if uses_scrub:
        base_texts = df["text_scrubbed"].tolist()
    else:
        base_texts = df["resume_text"].fillna("").astype(str).tolist()
    orig_preds = predict_batch(base_texts, model, tokenizer, device)

    if uses_scrub:
        print("  Scrubbing model — evaluating swaps after masking sensitive tokens")

    city_results = {}
    any_flip = np.zeros(len(df), dtype=bool)

    for swap_city in SWAP_CITIES:
        swapped_series = df["resume_text"].apply(
            lambda x: swap_cities_in_text(x, swap_city)
        )
        if uses_scrub:
            swapped = swapped_series.apply(scrub_text).tolist()
        else:
            swapped = swapped_series.fillna("").astype(str).tolist()

        swap_preds = predict_batch(swapped, model, tokenizer, device)
        flipped = (swap_preds != orig_preds)
        flip_rate = float(flipped.mean())
        any_flip |= flipped

        city_results[swap_city] = {
            "flip_rate": flip_rate,
            "num_flipped": int(flipped.sum()),
            "num_total": len(df),
        }
        print(f"  {swap_city}: {flip_rate:.3f} ({flipped.sum()}/{len(df)})")

        del swap_preds, flipped, swapped, swapped_series
        gc.collect()

    overall_flip = float(any_flip.mean())
    per_class = {}
    for i, cn in enumerate(class_names):
        mask = orig_preds == i
        if mask.sum() > 0:
            per_class[cn] = float(any_flip[mask].mean())

    y_true = le.transform(df["label"].map(l2s).fillna("generic_it_ops"))
    acc = float(accuracy_score(y_true, orig_preds))
    f1 = float(f1_score(y_true, orig_preds, average="macro"))

    all_results[model_name] = {
        "model_dir": model_subdir,
        "uses_scrubbing": uses_scrub,
        "accuracy": acc,
        "macro_f1": f1,
        "overall_flip_rate": overall_flip,
        "per_city_flip_rate": city_results,
        "per_class_flip_rate": per_class,
    }
    print(f"  Acc={acc:.3f}  F1={f1:.3f}  Flip={overall_flip:.3f}")

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
print(f"{'Model':<30} {'Acc':>7} {'F1':>7} {'Flip%':>8} {'Scrub':>6}")
print("-" * 58)
for name, res in all_results.items():
    if "error" in res:
        print(f"{name:<30} {'ERROR':>21}")
    else:
        scr = "Y" if res["uses_scrubbing"] else ""
        print(f"{name:<30} {res['accuracy']:>7.3f} {res['macro_f1']:>7.3f} {res['overall_flip_rate']:>8.3f} {scr:>6}")

In [ ]:
non_scrub = {k: v for k, v in all_results.items()
             if "error" not in v and not v["uses_scrubbing"]}

header = f"{'Model':<25}"
for city in SWAP_CITIES:
    header += f" {city[:10]:>12}"
print(header)
print("-" * (25 + 13 * len(SWAP_CITIES)))

for name, res in non_scrub.items():
    row = f"{name:<25}"
    for city in SWAP_CITIES:
        fr = res["per_city_flip_rate"].get(city, {}).get("flip_rate", 0)
        row += f" {fr:>12.3f}"
    print(row)

In [ ]:
for name, res in all_results.items():
    if "error" in res or res["uses_scrubbing"]:
        continue
    print(f"\n{name}:")
    per_class = res.get("per_class_flip_rate", {})
    for cls, fr in sorted(per_class.items(), key=lambda x: -x[1]):
        bar = "#" * int(fr * 50)
        print(f"  {cls:<35} {fr:.3f} {bar}")

In [ ]:
os.makedirs(str(OUTPUT_DIR), exist_ok=True)
out_path = OUTPUT_DIR / "city_swap_all_models.json"
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print(f"Saved: {out_path}")

In [ ]:
try:
    import matplotlib.pyplot as plt

    if non_scrub:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        names = list(non_scrub.keys())
        flips = [non_scrub[n]["overall_flip_rate"] for n in names]
        axes[0].barh(names, flips, color="steelblue")
        axes[0].set_xlabel("Overall Flip Rate")
        axes[0].set_title("City Swap: Flip Rate by Model")
        axes[0].axvline(x=0, color="red", linestyle="--", alpha=0.5)

        accs = [non_scrub[n]["accuracy"] for n in names]
        axes[1].scatter(accs, flips, s=100, c="steelblue", edgecolors="black")
        for i, name in enumerate(names):
            axes[1].annotate(name.split("_", 1)[1], (accs[i], flips[i]),
                           fontsize=8, ha="left", va="bottom")
        axes[1].set_xlabel("Accuracy")
        axes[1].set_ylabel("Flip Rate")
        axes[1].set_title("Accuracy vs City Bias")

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "city_swap_chart.png", dpi=150, bbox_inches="tight")
        plt.show()
except ImportError:
    print("matplotlib not installed, skipping charts")